# Autoresearch Adapter Demo

This notebook exercises the project-specific `autoresearch` adapter built on top of the local Ollama worker system.

It covers:

1. locating the `claw-code` control-plane repo and the `nodes/autoresearch-macos` repo
2. verifying Ollama reachability
3. running `autoresearch setup`
4. running one experiment packet through `autoresearch run`
5. parsing `run.log` through `autoresearch parse-log`

The notebook is designed to handle paths automatically.

In [1]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
import urllib.error
import urllib.request
from pathlib import Path

def find_claw_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "main.py").exists() and (candidate / "tests" / "test_porting_workspace.py").exists():
            return candidate
    raise RuntimeError(f"Could not locate claw-code root from {current}")

CLAW_ROOT = find_claw_root()
WORKSPACE_ROOT = CLAW_ROOT.parent.parent
AUTORESEARCH_ROOT = WORKSPACE_ROOT / "nodes" / "autoresearch-macos"
PYTHON = sys.executable

def run_cmd(args: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    cwd = cwd or CLAW_ROOT
    print("$", " ".join(shlex.quote(arg) for arg in args))
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        details = result.stderr.strip() or result.stdout.strip() or "no output"
        raise RuntimeError(f"command failed with exit code {result.returncode}: {details}")
    return result

def run_json(args: list[str], cwd: Path | None = None, check: bool = True):
    result = run_cmd(args, cwd=cwd, check=check)
    return json.loads(result.stdout)

def ollama_ready(host: str = "http://localhost:11434") -> bool:
    try:
        with urllib.request.urlopen(f"{host.rstrip('/')}/api/tags", timeout=2) as response:
            return response.status == 200
    except (urllib.error.URLError, TimeoutError):
        return False

print({
    "claw_root": str(CLAW_ROOT),
    "workspace_root": str(WORKSPACE_ROOT),
    "autoresearch_root": str(AUTORESEARCH_ROOT),
    "python": PYTHON,
})
if not AUTORESEARCH_ROOT.exists():
    raise RuntimeError(f"Autoresearch root does not exist: {AUTORESEARCH_ROOT}")

{'claw_root': '/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code', 'workspace_root': '/Users/wongdowling/Documents/autoresearch_harness', 'autoresearch_root': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos', 'python': '/Users/wongdowling/opt/anaconda3/bin/python'}


In [2]:
OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5-coder:7b"
OLLAMA_IS_READY = ollama_ready(OLLAMA_HOST)
print({"ollama_host": OLLAMA_HOST, "ollama_model": OLLAMA_MODEL, "ollama_ready": OLLAMA_IS_READY})
if not OLLAMA_IS_READY:
    raise RuntimeError("Ollama is not reachable. Start Ollama before running this notebook.")

{'ollama_host': 'http://localhost:11434', 'ollama_model': 'qwen2.5-coder:7b', 'ollama_ready': True}


In [3]:
packet_path = AUTORESEARCH_ROOT / "experiment.demo.json"
if not packet_path.exists():
    raise RuntimeError(f"Missing packet file: {packet_path}")
packet = json.loads(packet_path.read_text(encoding="utf-8"))
packet

{'objective': 'Try one scoped improvement to train.py aimed at reducing val_bpb while keeping the code simple.',
 'description': 'Inspect the current optimizer and architecture settings, then apply one small experimental change in train.py only.',
 'train_command': 'uv run train.py > run.log 2>&1',
 'timeout_seconds': 600,
 'log_path': 'run.log',
 'results_tsv': 'results.tsv',
 'syntax_check_command': 'python3 -m py_compile train.py'}

In [4]:
setup_result = run_json([
    PYTHON,
    "-m",
    "src.main",
    "autoresearch",
    "setup",
    "--root",
    str(AUTORESEARCH_ROOT),
])
setup_result

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main autoresearch setup --root /Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos
{
  "root": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos",
  "branch": "master",
  "data_ready": true,
  "tokenizer_ready": true,
  "results_tsv_path": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/results.tsv",
  "results_tsv_initialized": false,
  "cache_dir": "/Users/wongdowling/.cache/autoresearch",
  "suggested_tag": "apr07"
}



{'root': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos',
 'branch': 'master',
 'data_ready': True,
 'tokenizer_ready': True,
 'results_tsv_path': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/results.tsv',
 'results_tsv_initialized': False,
 'cache_dir': '/Users/wongdowling/.cache/autoresearch',
 'suggested_tag': 'apr07'}

In [5]:
run_result = run_json([
    PYTHON,
    "-m",
    "src.main",
    "autoresearch",
    "run",
    "--root",
    str(AUTORESEARCH_ROOT),
    "--packet",
    str(packet_path),
    "--model",
    OLLAMA_MODEL,
    "--host",
    OLLAMA_HOST,
    "--trace",
])
run_result["recommended_status"], run_result["experiment"]["success"] if run_result["experiment"] else None

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main autoresearch run --root /Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos --packet /Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/experiment.demo.json --model qwen2.5-coder:7b --host http://localhost:11434 --trace
{
  "setup": {
    "root": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos",
    "branch": "master",
    "data_ready": true,
    "tokenizer_ready": true,
    "results_tsv_path": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/results.tsv",
    "results_tsv_initialized": false,
    "cache_dir": "/Users/wongdowling/.cache/autoresearch",
    "suggested_tag": "apr07"
  },
  "worker": {
    "worker_id": "a8b99f54e6ce43d8b47d9c27f5e58003",
    "root": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos",
    "model": "qwen2.5-coder:7b",
    "host": "http://localhost:11434",
    "state": "finish

('discard', None)

In [6]:
print("recommended_status:", run_result["recommended_status"])
print("results_tsv:", run_result["results_tsv"])
print("worker state:", run_result["worker"]["state"])
print("worker tool calls:", run_result["worker"].get("last_result", {}).get("tool_calls", []))
print("changed_files:", run_result["worker"].get("last_result", {}).get("changed_files", []))
print("experiment:")
print(json.dumps(run_result["experiment"], indent=2))


recommended_status: discard
results_tsv: /Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/results.tsv
worker state: finished
worker tool calls: ['read_file']
changed_files: []
experiment:
null


In [9]:
parsed_log = run_json([
    PYTHON,
    "-m",
    "src.main",
    "autoresearch",
    "parse-log",
    "--root",
    str(AUTORESEARCH_ROOT),
    "run.log",
])
parsed_log

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main autoresearch parse-log --root /Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos run.log
{
  "success": false,
  "log_path": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/run.log",
  "timed_out": false,
  "return_code": 1,
  "val_bpb": null,
  "peak_vram_mb": null,
  "training_seconds": null,
  "total_seconds": null,
  "mfu_percent": null,
  "total_tokens_m": null,
  "num_steps": null,
  "num_params_m": null,
  "depth": null,
  "error": "run log did not contain a final val_bpb summary",
  "tail": "run logs below:"
}



{'success': False,
 'log_path': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/run.log',
 'timed_out': False,
 'return_code': 1,
 'val_bpb': None,
 'peak_vram_mb': None,
 'training_seconds': None,
 'total_seconds': None,
 'mfu_percent': None,
 'total_tokens_m': None,
 'num_steps': None,
 'num_params_m': None,
 'depth': None,
 'error': 'run log did not contain a final val_bpb summary',
 'tail': 'run logs below:'}

## Success Criteria

This notebook is a successful smoke test when:

- `autoresearch setup` returns JSON with the expected repo root
- `autoresearch run` returns JSON
- `run_result["worker"]["last_result"]["tool_calls"]` is non-empty
- `run_result["experiment"]` is present
- `results.tsv` exists in the autoresearch repo
- `autoresearch parse-log` returns structured metrics
